# 1) Create source sound collection

This notebook includes the code to create the collection of sounds that will later be used as source material for our audio mosaicing application. The collection of sounds is created by defining a number of queries to be performed using the Freesound API and concatenanting the results of each query. A number of metadata fields are stored for each sound in the collection and saved into a Pandas DataFrame object and CSV file in disk. For each sound in the collection, we also download an OGG preview and store it in disk.

This notebook uses the `freesound` Python package for interacting with the Freesound API. The source code for this package can be found here: https://github.com/mtg/freesound-python. In this repository you'll find a Python script with [examples](https://github.com/MTG/freesound-python/blob/master/examples.py) to learn how to interact with the API. Nevertheless, if you are further interested in the Freesound API, check the [API documentation](http://freesound.org/docs/api/) which provides more information.

**NOTE**: A Freesound API key is provided in this notebook, but you should make a Freesound account and get your own key. You can get a key here: https://freesound.org/apiv2/apply/

In [2]:
# Essentia
!pip install essentia
# Freesound-python
!pip install git+https://github.com/mtg/freesound-python.git
# Mount drive and cd to notebook folder
# from google.colab import drive
# drive.mount('/content/drive')


  Cloning https://github.com/mtg/freesound-python.git to /private/var/folders/3f/r_xskjpj7zz615kg232x7wjw0000gn/T/pip-req-build-5yp57yis
  Running command git clone --filter=blob:none --quiet https://github.com/mtg/freesound-python.git /private/var/folders/3f/r_xskjpj7zz615kg232x7wjw0000gn/T/pip-req-build-5yp57yis
  Resolved https://github.com/mtg/freesound-python.git to commit 73cf6d14f7ce8174d943bdc78ff30f99878a5db8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# Replace with your path
%ls
#%cd '/content/drive/MyDrive/SMC/ACTSM2026/Block II/Session 1 - Sound retrieval with Freesound'

1 - Create source sound collection.ipynb
2 - Analyze source collection and target file.ipynb
213524__garzul__120-bpm-distorded-drum-loop.wav
213524__garzul__120-bpm-distorded-drum-loop.wav.reconstructed.wav
262350__stereo-surgeon__grinder-drum-loop.wav
3 - Reconstruct target file with source collection frames.ipynb
README.md
Sound retrieval with Freesound and creative applications.pptx
dataframe.csv
dataframe_source.csv
dataframe_target.csv
files/
mosaicing_examples_28x28/


In [3]:
import os
import pandas as pd
import numpy as np
import freesound
from IPython.display import display

FREESOUND_API_KEY = 'sdDO8Z7lOFji4eGwy9m2llENzXgQmqduSOLpAZ3z'  # Please replace by your own Freesound API key
FILES_DIR = 'files'  # Place where to store the downloaded diles. Will be relative to the current folder.
DATAFRAME_FILENAME = 'dataframe.csv'  # File where we'll store the metadata of our sounds collection
FREESOUND_STORE_METADATA_FIELDS = ['id', 'name', 'username', 'previews', 'license', 'tags']  # Freesound metadata properties to store

freesound_client = freesound.FreesoundClient()
freesound_client.set_token(FREESOUND_API_KEY)
if not os.path.exists(FILES_DIR): os.mkdir(FILES_DIR)

In [6]:
# Define some util functions

def query_freesound(query, filter, num_results=10):
    """Queries freesound with the given query and filter values.
    If no filter is given, a default filter is added to only get sounds shorter than 30 seconds.
    """
    if filter is None:
        filter = 'duration:[0 TO 30]'  # Set default filter
    pager = freesound_client.search(
        query = query,
        filter = filter,
        fields = ','.join(FREESOUND_STORE_METADATA_FIELDS),
        group_by_pack = 1,
        page_size = num_results
    )
    return [sound for sound in pager]

def retrieve_sound_preview(sound, directory):
    """Download the high-quality OGG sound preview of a given Freesound sound object to the given directory.
    """
    return freesound.FSRequest.retrieve(
        sound.previews.preview_hq_ogg,
        freesound_client,
        os.path.join(directory, sound.previews.preview_hq_ogg.split('/')[-1])
    )

def make_pandas_record(sound):
    """Create a dictionary with the metadata that we want to store for each sound.
    """
    record = {key: sound.as_dict()[key] for key in FREESOUND_STORE_METADATA_FIELDS}
    del record['previews']  # Don't store previews dict in record
    record['freesound_id'] = record['id']  # Rename 'id' to 'freesound_id'
    del record['id']
    record['path'] = "files/" + sound.previews.preview_hq_ogg.split("/")[-1]  # Store path of downloaded file
    return record

In [8]:
# Build our collection of sounds

# Our collection of sounds is made by appending the results of a number of different queries to freesound
# The query terms, query filters and the number of results per query are all defined here.
# Information about how to define filters can be found in the Freesound API documentation: https://freesound.org/docs/api/resources_apiv2.html#request-parameters-text-search-parameters
freesound_queries = [
    {
        'query': 'race car',
        'filter': None,
        'num_results': 20,
    },
    {
        'query': 'car crash',
        'filter': 'duration:[0 TO 5]',
        'num_results': 20,
    },
    {
        'query': 'car horns',
        'filter': 'duration:[20 TO 60]',
        'num_results': 20,
    },
]

# Do all queries and concatenate the results in a single list of sounds
sounds = sum([query_freesound(query['query'], query['filter'], query['num_results']) for query in freesound_queries],[])

# Download the sounds and save them to FILES_DIR folder
for count, sound in enumerate(sounds):
    print('Downloading sound with id {0} [{1}/{2}]'.format(sound.id, count + 1, len(sounds)))
    retrieve_sound_preview(sound, 'files/')

# Make a Pandas DataFrame with the metadata of our sound collection and save it
df =  pd.DataFrame([make_pandas_record(s) for s in sounds])
df.to_csv(DATAFRAME_FILENAME)
print('Saved DataFrame with {0} entries! {1}'.format(len(df), DATAFRAME_FILENAME))

# Show the contents of our DataFrame (the metadata of our source collection)
display(df)

Saved DataFrame with 60 entries! dataframe.csv


,name,username,license,tags,freesound_id,path
0,"Car race, Nordschleife, VLN, single car passin...",Dominik_W,http://creativecommons.org/publicdomain/zero/1.0/,"[Germany, Nordschleife, VLN, car, engines, fie...",350674,files/350674_4162229-hq.ogg
1,Derby car_race B,snapssound,https://creativecommons.org/licenses/by/4.0/,"[Car, Drive, Engine, Fast, Motor, Passing, Race]",700476,files/700476_9771749-hq.ogg
2,"TOONVeh-Blue Snowball Microphone, CU_Kazoo, Im...",designerschoice,http://creativecommons.org/publicdomain/zero/1.0/,"[Blue-Snowball, Blue-brand, car, cartoon, engi...",805257,files/805257_6951162-hq.ogg
3,space race car.mp3,IFartInUrGeneralDirection,https://creativecommons.org/licenses/by/4.0/,"[aliens, car, nascar, race, ship, space]",58260,files/58260_384275-hq.ogg
4,space race car pit stop.mp3,IFartInUrGeneralDirection,https://creativecommons.org/licenses/by/4.0/,"[aliens, car, nascar, race, ship, space]",58259,files/58259_384275-hq.ogg
5,Racing Car,alegemaate,http://creativecommons.org/publicdomain/zero/1.0/,"[car, drive, engine, motor, race, racing, synt...",413658,files/413658_2531187-hq.ogg
6,BCR F1 Car 19.wav,LG,https://creativecommons.org/licenses/by/4.0/,"[alonso, bavaria, car, cars, city, f1, fernand...",77919,files/77919_36188-hq.ogg
7,8-bit Racing Car,Breviceps,http://creativecommons.org/publicdomain/zero/1.0/,"[8-bit, 8bit, accelerate, arcade, bit, c64, ca...",467250,files/467250_9159316-hq.ogg
8,space race car 3.mp3,IFartInUrGeneralDirection,https://creativecommons.org/licenses/by/4.0/,"[aliens, beat, car, explosion, fart, nascar, r...",58257,files/58257_384275-hq.ogg
9,05995 rapid car arrival.wav,Robinhood76,https://creativecommons.org/licenses/by-nc/4.0/,"[arrival, car, fast, race, racing, rapid, spee...",317458,files/317458_321967-hq.ogg


Inspired by the famous video in which some F1 mechanics make a car engine play God Save The Queen, we are going to select a few car sounds and try to turn them into music. The goal is to see whether we can do something similar or interesting in that direction. For that, we will also select some car horns and some percusive sounds (car crashes)